<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/data_prep/01_Slakh2100_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================
# [Bass Separator] Ultimate Environment Setup (v4.0)
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 통합 환경 설정을 시작합니다...")

# 1. Google Drive 마운트
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. GitHub 최신화 및 경로 설정
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# 3. 커스텀 모듈 실행
try:
    from src.env_setup import init_colab_env
    init_colab_env()
    # 주의: 이 노트북은 대규모 전처리용이므로 load_data_from_drive()는 생략합니다.
except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")

# 전처리에 필요한 필수 패키지 설치 (requirements.txt에 없다면 대비용)
!pip install -q soundfile pyyaml tqdm

print("\n🎉 Ready to Rock! 환경 셋업 완료.")


In [ ]:
# ==================================================================
# [DataOps] Slakh2100-redux 원본 데이터 다운로드 및 압축 해제
# ==================================================================
import os

RAW_DATA_DIR = "/content/slakh_raw"
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.chdir(RAW_DATA_DIR)

print("📥 Slakh2100-redux 데이터를 준비합니다...")

# 시나리오 A: 원본 압축 파일이 구글 드라이브에 이미 있는 경우 (권장)
# DRIVE_ZIP_PATH = "/content/drive/MyDrive/Bass_separator/raw_data/slakh2100-redux.zip"
# !unzip -q "{DRIVE_ZIP_PATH}" -d "{RAW_DATA_DIR}"

# 시나리오 B: 직접 다운로드하는 경우 (Zenodo 등에서 제공하는 링크가 있을 때)
# !wget -c "SLAKH_REDUX_DOWNLOAD_URL" -O slakh.zip
# !unzip -q slakh.zip -d "{RAW_DATA_DIR}"

print("✅ 압축 해제 완료. (경로: /content/slakh_raw/)")
os.chdir(PROJECT_PATH) # 작업 디렉토리 복귀


In [ ]:
# ==================================================================
# [DataOps] 데이터 파싱 및 평가용 데이터셋(slakh_eval) 구축
# ==================================================================
import os
import yaml
import shutil
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm

# 경로 설정
SOURCE_DIR = "/content/slakh_raw/train" # 압축 푼 폴더 구조에 맞게 수정 필요
OUTPUT_DIR = "/content/slakh_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_slakh_track(source_path: Path, output_dir: str):
    meta_path = source_path / "metadata.yaml"
    if not meta_path.exists(): return False

    with open(meta_path, 'r') as f:
        metadata = yaml.safe_load(f)

    bass_stem = None
    other_stems = []

    # 1. 베이스 트랙 식별
    for stem_name, stem_info in metadata['stems'].items():
        prog_num = stem_info['program_num']
        if (32 <= prog_num <= 39) or prog_num == 43:
            if bass_stem is None: bass_stem = stem_name
            else: return False # 베이스가 2개 이상이면 변수 통제를 위해 스킵
        else:
            other_stems.append(stem_name)

    if not bass_stem: return False

    out_track_dir = Path(output_dir) / source_path.name
    out_track_dir.mkdir(parents=True, exist_ok=True)

    # 2. GT(Ground Truth) 파일 복사
    shutil.copy(source_path / "stems" / f"{bass_stem}.wav", out_track_dir / "bass_gt.wav")
    shutil.copy(source_path / "MIDI" / f"{bass_stem}.mid", out_track_dir / "bass_gt.mid")

    # mix.wav 파일이 존재할 경우 복사 (존재하지 않으면 패스)
    mix_path = source_path / "mix.wav"
    if mix_path.exists():
        shutil.copy(mix_path, out_track_dir / "mix.wav")

    # 3. Bassless MR 믹스다운
    mix_audio, target_sr = None, None
    for stem in other_stems:
        stem_path = source_path / "stems" / f"{stem}.wav"
        if not stem_path.exists(): continue
        audio, sr = sf.read(stem_path)
        if mix_audio is None:
            mix_audio = np.zeros_like(audio)
            target_sr = sr
        mix_audio += audio

    # Clipping 방지 정규화
    if mix_audio is not None:
        max_amp = np.max(np.abs(mix_audio))
        if max_amp > 1.0: mix_audio /= max_amp
        sf.write(out_track_dir / "bassless_mr.wav", mix_audio, target_sr)

    return True

# 실행
track_dirs = [d for d in Path(SOURCE_DIR).iterdir() if d.is_dir()]
success_count = 0

print(f"🔍 총 {len(track_dirs)}개의 트랙 검사를 시작합니다...")
for track_dir in tqdm(track_dirs):
    if process_slakh_track(track_dir, OUTPUT_DIR):
        success_count += 1

print(f"🎉 전처리 완료! 총 {success_count}개의 유효한 평가용 트랙이 '{OUTPUT_DIR}'에 생성되었습니다.")


In [ ]:
# ==================================================================
# [DataOps] 전처리 완료 데이터 구글 드라이브로 영구 저장
# ==================================================================
import shutil

DRIVE_TARGET_DIR = "/content/drive/MyDrive/Bass_separator/datasets/slakh_eval"
LOCAL_EVAL_DIR = "/content/slakh_eval"

print(f"💾 Colab 디스크의 전처리 결과를 구글 드라이브로 복사합니다...")
print(f"Source: {LOCAL_EVAL_DIR}")
print(f"Target: {DRIVE_TARGET_DIR}")

# dirs_exist_ok=True를 사용하여 기존 폴더에 덮어쓰기 허용 (Python 3.8+)
shutil.copytree(LOCAL_EVAL_DIR, DRIVE_TARGET_DIR, dirs_exist_ok=True)

print("✅ 성공적으로 드라이브에 저장되었습니다! 이제 향후 실험(평가) 시 이 경로를 참조하면 됩니다.")
